In [ ]:
from heapq import heappush, heappop
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, FrozenSet, Any

class Cell:
    def __init__(self):
        self.walls = {'N': False, 'E': False, 'S': False, 'W': False}
        self.is_gate = None

# Directions
# 0: N, 1: E, 2: S, 3: W
DIRECTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
DIR_LABELS = ['N', 'E', 'S', 'W']

# Costs
MOVE_COST = 1.21
HEAVY_MOVE_COST = 1.45  # pushing a bottle
TURN_COST = 0.65
UTURN_COST = 2 * TURN_COST
ATTACH_COST = 0.0
DETACH_COST = 0.0
VISIT_GATE_COST = 0.0  # explicit gate count with no bottle

# Settings
REQUIRE_ALL_BOTTLES_IN_GATES = True

# False: forward/backward treated equally
# True: backward gets a big penalty (but is still allowed)
PREFER_FORWARD = False
BACKWARD_PENALTY_NORMAL = 2.50
BACKWARD_PENALTY_ESCAPE = 0.05

# Bottle state:
# ('e', r, c, dir) bottle blocks an edge, anchored to (r,c) on edge dir
# ('c', r, c)      bottle is co-located with robot inside cell (r,c) while pushing
BottleState = Tuple[Any, ...]


def is_in_bounds(r: int, c: int, rows: int, cols: int) -> bool:
    return 0 <= r < rows and 0 <= c < cols


def opposite_dir_label(lbl: str) -> str:
    return {'N': 'S', 'S': 'N', 'E': 'W', 'W': 'E'}[lbl]


def allowed_cells_default(rows: int, cols: int) -> FrozenSet[Tuple[int, int]]:
    # inner track cells: r=1..rows-2, c=1..cols-2
    return frozenset((r, c) for r in range(1, rows - 1) for c in range(1, cols - 1))


def bottle_blocks_edge(
    r: int,
    c: int,
    dir_label: str,
    bottle_state: Dict[int, BottleState],
    remaining: Tuple[int, ...],
    ignore_bid: Optional[int] = None
) -> Optional[int]:
    rem = set(remaining)
    if ignore_bid is not None:
        rem.discard(ignore_bid)

    # anchored on this cell side
    for bid, st in bottle_state.items():
        if bid in rem and st[0] == 'e' and (st[1], st[2], st[3]) == (r, c, dir_label):
            return bid

    # anchored on neighbor cell opposite side
    d = DIR_LABELS.index(dir_label)
    dr, dc = DIRECTIONS[d]
    nr, nc = r + dr, c + dc
    opp = opposite_dir_label(dir_label)

    for bid, st in bottle_state.items():
        if bid in rem and st[0] == 'e' and (st[1], st[2], st[3]) == (nr, nc, opp):
            return bid

    return None


def wall_blocks_move(
    r_from: int,
    c_from: int,
    r_to: int,
    c_to: int,
    maze: List[List[Cell]],
    blocked_cells: FrozenSet[Tuple[int, int]],
    start: Tuple[int, int],
    end: Tuple[int, int],
    allowed_cells: FrozenSet[Tuple[int, int]]
) -> bool:
    rows, cols = len(maze), len(maze[0])

    # Disallow entering the outer ring (except if start/end ever placed there)
    if is_in_bounds(r_to, c_to, rows, cols):
        if (r_to, c_to) not in allowed_cells and (r_to, c_to) != start and (r_to, c_to) != end:
            return True

    # bounds
    if not is_in_bounds(r_from, c_from, rows, cols) and (r_from, c_from) != start:
        return True
    if not is_in_bounds(r_to, c_to, rows, cols) and (r_to, c_to) != end:
        return True

    # resting bottle blocks re-entry
    if (r_to, c_to) in blocked_cells:
        return True

    cell_from = maze[r_from][c_from] if is_in_bounds(r_from, c_from, rows, cols) else None
    cell_to = maze[r_to][c_to] if is_in_bounds(r_to, c_to, rows, cols) else None

    dr = r_to - r_from
    dc = c_to - c_from
    if dr == -1 and dc == 0:
        dir_from, dir_to = 'N', 'S'
    elif dr == 1 and dc == 0:
        dir_from, dir_to = 'S', 'N'
    elif dr == 0 and dc == 1:
        dir_from, dir_to = 'E', 'W'
    elif dr == 0 and dc == -1:
        dir_from, dir_to = 'W', 'E'
    else:
        return True

    if cell_from and cell_from.walls.get(dir_from, False):
        return True
    if cell_to and cell_to.walls.get(dir_to, False):
        return True
    return False


@dataclass(frozen=True)
class State:
    pos: Tuple[int, int]
    direction: int
    remaining: Tuple[int, ...]                 # bottles not yet deposited
    gate_items: Tuple[Tuple[str, bool], ...]   # gates counted
    blocked_cells: FrozenSet[Tuple[int, int]]  # cells containing resting bottles
    bottle_items: Tuple[Tuple[int, BottleState], ...]
    pushing: Optional[int]
    pushed_once: bool
    must_back_out: bool                        # after deposit, force one backward move

    def gates(self) -> Dict[str, bool]:
        return dict(self.gate_items)

    def bottle_state(self) -> Dict[int, BottleState]:
        return dict(self.bottle_items)


def make_state(
    pos: Tuple[int, int],
    direction: int,
    remaining: Tuple[int, ...],
    gates: Dict[str, bool],
    blocked_cells: FrozenSet[Tuple[int, int]],
    bottle_state: Dict[int, BottleState],
    pushing: Optional[int],
    pushed_once: bool,
    must_back_out: bool
) -> State:
    return State(
        pos=pos,
        direction=direction,
        remaining=remaining,
        gate_items=tuple(sorted(gates.items())),
        blocked_cells=blocked_cells,
        bottle_items=tuple(sorted(bottle_state.items())),
        pushing=pushing,
        pushed_once=pushed_once,
        must_back_out=must_back_out
    )


def is_gate_cell(maze: List[List[Cell]], rc: Tuple[int, int]) -> bool:
    r, c = rc
    rows, cols = len(maze), len(maze[0])
    if not is_in_bounds(r, c, rows, cols):
        return False
    return maze[r][c].is_gate is not None


def heuristic(s: State, gates_pos: Dict[str, Tuple[int, int]], end: Tuple[int, int], require_all_bottles: bool) -> float:
    r, c = s.pos
    visited = dict(s.gate_items)

    lb = abs(r - end[0]) + abs(c - end[1])
    for g, (gr, gc) in gates_pos.items():
        if not visited.get(g, False):
            lb = max(lb, abs(r - gr) + abs(c - gc))

    h = lb * MOVE_COST
    if require_all_bottles:
        h += len(s.remaining) * HEAVY_MOVE_COST
    return h


def get_next_states(
    s: State,
    maze: List[List[Cell]],
    gates_pos: Dict[str, Tuple[int, int]],
    start: Tuple[int, int],
    end: Tuple[int, int],
    allowed_cells: FrozenSet[Tuple[int, int]],
    prefer_forward: bool
) -> List[Tuple[str, float, State]]:
    out: List[Tuple[str, float, State]] = []
    r, c = s.pos
    d = s.direction
    gates = dict(s.gate_items)
    bs = s.bottle_state()
    blocked = s.blocked_cells

    # If we must back out, only allow one backward step
    if s.must_back_out:
        if s.pushing is not None:
            return []

        dr, dc = DIRECTIONS[d]
        br, bc = r - dr, c - dc
        back_label = DIR_LABELS[(d + 2) % 4]

        if bottle_blocks_edge(r, c, back_label, bs, s.remaining) is None:
            if not wall_blocks_move(r, c, br, bc, maze, blocked, start, end, allowed_cells):
                penalty = BACKWARD_PENALTY_ESCAPE if prefer_forward else 0.0
                out.append((
                    'B',
                    MOVE_COST + penalty,
                    make_state((br, bc), d, s.remaining, gates, blocked, bs, None, False, False)
                ))
        return out

    # Count a gate with no bottle: explicit G action
    if s.pushing is None and is_gate_cell(maze, (r, c)):
        gl = maze[r][c].is_gate
        if not gates.get(gl, False):
            g2 = gates.copy()
            g2[gl] = True
            out.append(('G', VISIT_GATE_COST, make_state((r, c), d, s.remaining, g2, blocked, bs.copy(), None, False, False)))

    # Turns
    for lab, nd, cost in [('L', (d - 1) % 4, TURN_COST),
                          ('R', (d + 1) % 4, TURN_COST),
                          ('U', (d + 2) % 4, UTURN_COST)]:
        if s.pushing is not None and lab == 'U' and not s.pushed_once:
            continue

        if s.pushing is not None:
            bid = s.pushing
            # cannot turn if bottle still edge-anchored
            if bs.get(bid, (None,))[0] == 'e':
                continue
            out.append((lab, cost, make_state((r, c), nd, s.remaining, gates.copy(), blocked, bs.copy(), bid, s.pushed_once, False)))
        else:
            out.append((lab, cost, make_state((r, c), nd, s.remaining, gates.copy(), blocked, bs.copy(), None, False, False)))

    # Forward delta
    fwd_label = DIR_LABELS[d]
    dr, dc = DIRECTIONS[d]
    nr, nc = r + dr, c + dc

    # Attach
    if s.pushing is None:
        bid = bottle_blocks_edge(r, c, fwd_label, bs, s.remaining)
        if bid is not None:
            out.append(('P', ATTACH_COST, make_state((r, c), d, s.remaining, gates.copy(), blocked, bs.copy(), bid, False, False)))

    # Detach deposit (only in a gate cell)
    if s.pushing is not None:
        bid = s.pushing
        if bs.get(bid, (None,))[0] == 'c' and is_gate_cell(maze, (r, c)):
            gl = maze[r][c].is_gate
            nb = bs.copy()
            nb.pop(bid, None)
            rem2 = tuple(x for x in s.remaining if x != bid)

            new_block = set(blocked)
            new_block.add((r, c))

            g2 = gates.copy()
            g2[gl] = True  # deposit counts the gate

            out.append(('D', DETACH_COST, make_state((r, c), d, rem2, g2, frozenset(new_block), nb, None, False, True)))

    # Forward move
    if s.pushing is None:
        if bottle_blocks_edge(r, c, fwd_label, bs, s.remaining) is None:
            if not wall_blocks_move(r, c, nr, nc, maze, blocked, start, end, allowed_cells):
                out.append(('F', MOVE_COST, make_state((nr, nc), d, s.remaining, gates.copy(), blocked, bs.copy(), None, False, False)))
    else:
        bid = s.pushing
        # FIX: only block if the edge we cross is blocked by a different bottle
        blocking = bottle_blocks_edge(r, c, fwd_label, bs, s.remaining, ignore_bid=bid)
        if blocking is None:
            if not wall_blocks_move(r, c, nr, nc, maze, blocked, start, end, allowed_cells):
                nb = bs.copy()
                nb[bid] = ('c', nr, nc)
                out.append(('F', HEAVY_MOVE_COST, make_state((nr, nc), d, s.remaining, gates.copy(), blocked, nb, bid, True, False)))

    # Backward (not pushing)
    if s.pushing is None:
        back_label = DIR_LABELS[(d + 2) % 4]
        br, bc = r - dr, c - dc

        if bottle_blocks_edge(r, c, back_label, bs, s.remaining) is None:
            if not wall_blocks_move(r, c, br, bc, maze, blocked, start, end, allowed_cells):
                penalty = 0.0
                if prefer_forward:
                    penalty = BACKWARD_PENALTY_ESCAPE if (is_gate_cell(maze, (r, c)) or ((r, c) in blocked)) else BACKWARD_PENALTY_NORMAL
                out.append(('B', MOVE_COST + penalty, make_state((br, bc), d, s.remaining, gates.copy(), blocked, bs.copy(), None, False, False)))

    return out


def find_optimal_path(
    maze: List[List[Cell]],
    start: Tuple[int, int],
    end: Tuple[int, int],
    gates_pos: Dict[str, Tuple[int, int]],
    bottles: Dict[int, Tuple[int, int, str]],
    initial_direction: int,
    prefer_forward: bool,
    require_all_bottles: bool
):
    # init bottle states as edge-anchored
    bs = {bid: ('e', r, c, dlbl) for bid, (r, c, dlbl) in bottles.items()}

    gates = {g: False for g in gates_pos}
    remaining = tuple(sorted(bottles.keys()))

    allowed_cells = allowed_cells_default(len(maze), len(maze[0]))
    allowed_cells = frozenset(set(allowed_cells) | {start, end})

    start_state = make_state(start, initial_direction, remaining, gates, frozenset(), bs, None, False, False)

    def is_goal(st: State) -> bool:
        if st.pos != end:
            return False
        if st.pushing is not None:
            return False
        if st.must_back_out:
            return False
        if require_all_bottles and len(st.remaining) != 0:
            return False
        return all(dict(st.gate_items).values())

    pq = []
    came_from: Dict[State, State] = {}
    move_taken: Dict[State, str] = {}
    g_score: Dict[State, float] = {start_state: 0.0}
    counter = 0

    heappush(pq, (heuristic(start_state, gates_pos, end, require_all_bottles), 0.0, counter, start_state))
    counter += 1

    while pq:
        _, cur_g, _, cur = heappop(pq)
        if cur_g != g_score.get(cur, None):
            continue

        if is_goal(cur):
            actions: List[str] = []
            states: List[State] = [cur]
            st = cur
            while st in came_from:
                actions.append(move_taken[st])
                st = came_from[st]
                states.append(st)
            actions.reverse()
            states.reverse()
            return actions, states, cur_g

        for action, cost, nxt in get_next_states(cur, maze, gates_pos, start, end, allowed_cells, prefer_forward):
            tg = cur_g + cost
            if tg < g_score.get(nxt, 1e18):
                g_score[nxt] = tg
                came_from[nxt] = cur
                move_taken[nxt] = action
                counter += 1
                heappush(pq, (tg + heuristic(nxt, gates_pos, end, require_all_bottles), tg, counter, nxt))

    return None, None, None


def actions_to_robot(actions: List[str]) -> List[str]:
    out: List[str] = []
    for a in actions:
        if a == 'U':
            out.extend(['R', 'R'])
        elif a in ('F', 'B', 'L', 'R'):
            out.append(a)
        # P, D, G are internal
    return out


def summarize_order(maze: List[List[Cell]], actions: List[str], states: List[State]) -> Tuple[List[str], List[Tuple[int, str]]]:
    events: List[str] = []
    deposits: List[Tuple[int, str]] = []

    for i in range(1, len(states)):
        prev = states[i - 1]
        cur = states[i]

        removed = set(prev.remaining) - set(cur.remaining)
        if removed:
            bid = sorted(list(removed))[0]
            r, c = cur.pos
            gate = maze[r][c].is_gate if is_gate_cell(maze, (r, c)) else '?'
            deposits.append((bid, gate))
            events.append(f"bottle {bid} to gate {gate}")
            continue

        prevg = dict(prev.gate_items)
        curg = dict(cur.gate_items)
        for g in curg:
            if (not prevg.get(g, False)) and curg.get(g, False):
                events.append(f"gate {g}")

    return events, deposits


def print_grid(maze, start, end, bottles, blocked_cells=frozenset()):
    rows, cols = len(maze), len(maze[0])

    # map cell -> bottle id string, based on anchor cell in bottles dict
    bottle_at_cell: Dict[Tuple[int, int], str] = {}
    for bid, (br, bc, _dir) in bottles.items():
        if (br, bc) not in bottle_at_cell:
            bottle_at_cell[(br, bc)] = str(bid)
        else:
            # keep smallest id if multiple anchor on same cell
            try:
                prev = int(bottle_at_cell[(br, bc)])
                bottle_at_cell[(br, bc)] = str(min(prev, bid))
            except Exception:
                bottle_at_cell[(br, bc)] = str(bid)

    min_row = min(0, start[0])
    max_row = max(rows - 1, start[0])
    min_col = min(0, start[1])
    max_col = max(cols - 1, start[1])

    print("Grid Coordinates:")
    for rr in range(min_row, max_row + 1):
        for cc in range(min_col, max_col + 1):
            print(f"({rr},{cc})", end=' ')
        print()
    print()

    for rr in range(min_row, max_row + 1):
        cell_line = ''
        for cc in range(min_col, max_col + 1):
            if (0 <= rr < rows) and (0 <= cc < cols):
                cell = maze[rr][cc]

                if (rr, cc) == start:
                    cell_char = 'S'
                elif (rr, cc) == end:
                    cell_char = 'E'
                elif (rr, cc) in blocked_cells:
                    cell_char = 'X'
                elif cell.is_gate is not None:
                    cell_char = cell.is_gate
                elif (rr, cc) in bottle_at_cell:
                    cell_char = bottle_at_cell[(rr, cc)]
                else:
                    cell_char = '.'

                cell_line += f"{cell_char}"
                if cc < max_col:
                    if cc < cols - 1 and cell.walls['E']:
                        cell_line += '|'
                    else:
                        cell_line += ' '
            else:
                cell_line += ' '
                if cc < max_col:
                    cell_line += ' '
        print(cell_line)

        if rr < max_row:
            wall_line = ''
            for cc in range(min_col, max_col + 1):
                if (0 <= rr < rows - 1) and (0 <= cc < cols):
                    cell = maze[rr][cc]
                    wall_line += ('— ' if cell.walls['S'] else '  ')
                else:
                    wall_line += '  '
            print(wall_line)
    print()


def create_maze():
    rows, cols = 6, 7
    maze = [[Cell() for _ in range(cols)] for _ in range(rows)]

    #(1,1) (1,2) (1,3) (1,4) (1,5)
    #(2,1) (2,2) (2,3) (2,4) (2,5)
    #(3,1) (3,2) (3,3) (3,4) (3,5)
    #(4,1) (4,2) (4,3) (4,4) (4,5)

    # Gates positions
    gates = {
        'A': (1, 5),
        'B': (4, 5),
        'C': (1, 1),
        'D': (4, 1),
        'E': (4, 3),
        'F': (2, 4)
    }

    #(0,0) (0,1) (0,2) (0,3) (0,4) (0,5) (0,6)
    #(1,0) (1,1) (1,2) (1,3) (1,4) (1,5) (1,6)
    #(2,0) (2,1) (2,2) (2,3) (2,4) (2,5) (2,6)
    #(3,0) (3,1) (3,2) (3,3) (3,4) (3,5) (3,6)
    #(4,0) (4,1) (4,2) (4,3) (4,4) (4,5) (4,6)
    #(5,0) (5,1) (5,2) (5,3) (5,4) (5,5) (5,6)

    barriers = [
        # These are the outside ones that prevent the robot from going into the "outer square".
        # Comment out the one from where the robot has to enter from.
        ((0, 1), 'S'),
        ((0, 2), 'S'),
        ((0, 3), 'S'),
        ((0, 4), 'S'),
        ((0, 5), 'S'),
        ((5, 1), 'N'),
        ((5, 2), 'N'),
        ((5, 3), 'N'),
        #((5, 4), 'N'),
        ((5, 5), 'N'),
        ((1, 0), 'E'),
        ((2, 0), 'E'),
        ((3, 0), 'E'),
        ((4, 0), 'E'),
        ((1, 6), 'W'),
        ((2, 6), 'W'),
        ((3, 6), 'W'),
        ((4, 6), 'W'),
        # End of the outside walls

        # Inside barriers
        ((1, 2), 'S'),
        ((1, 3), 'S'),
        ((1, 4), 'S'),
        ((1, 4), 'E'),
        ((2, 3), 'E'),
        ((2, 4), 'E'),
        ((4, 1), 'N'),
        ((4, 3), 'E'),
        ((4, 3), 'N'),
        ((4, 4), 'E')
    ]

    for (rr, cc), dirc in barriers:
        maze[rr][cc].walls[dirc] = True
        if dirc == 'N' and rr > 0:
            maze[rr - 1][cc].walls['S'] = True
        if dirc == 'E' and cc < cols - 1:
            maze[rr][cc + 1].walls['W'] = True
        if dirc == 'S' and rr < rows - 1:
            maze[rr + 1][cc].walls['N'] = True
        if dirc == 'W' and cc > 0:
            maze[rr][cc - 1].walls['E'] = True

    # mark each gate cell
    for label, (gr, gc) in gates.items():
        maze[gr][gc].is_gate = label

    # bottle blocks an edge between two cells, anchored to one side
    bottles = {
        0: (3, 1, 'E'),
        1: (1, 2, 'E'),
        2: (1, 3, 'E'),
        3: (2, 2, 'E'),
        4: (3, 4, 'E'),
    }

    start = (4, 4)
    end = (1, 4)
    initial_direction = 0
    return maze, start, end, gates, bottles, initial_direction


if __name__ == '__main__':
    maze, start, end, gates, bottles, initial_direction = create_maze()

    print('Maze layout:')
    print_grid(maze, start, end, bottles)

    actions, states, total_time = find_optimal_path(
        maze, start, end, gates, bottles, initial_direction,
        prefer_forward=PREFER_FORWARD,
        require_all_bottles=REQUIRE_ALL_BOTTLES_IN_GATES
    )

    if actions is None:
        print('No valid path found.')
    else:
        print('Internal actions:')
        print(' '.join(actions))
        print()

        robot_cmds = actions_to_robot(actions)
        print('Robot commands:')
        print(' '.join(robot_cmds))
        print()

        events, deposits = summarize_order(maze, actions, states)
        print('Order summary:')
        print(', '.join(events))
        print()

        print('Bottle to gate mapping:')
        for bid, gl in deposits:
            print(f'bottle {bid} to gate {gl}')
        print()

        print(f'Total time: {total_time:.2f}')
        print()

        print(f'Forward movements: {robot_cmds.count("F")}')
        print(f'Backward movements: {robot_cmds.count("B")}')
        print(f'Right turns: {robot_cmds.count("R")}')
        print(f'Left turns: {robot_cmds.count("L")}')


Maze layout:
Grid Coordinates:
(0,0) (0,1) (0,2) (0,3) (0,4) (0,5) (0,6) 
(1,0) (1,1) (1,2) (1,3) (1,4) (1,5) (1,6) 
(2,0) (2,1) (2,2) (2,3) (2,4) (2,5) (2,6) 
(3,0) (3,1) (3,2) (3,3) (3,4) (3,5) (3,6) 
(4,0) (4,1) (4,2) (4,3) (4,4) (4,5) (4,6) 
(5,0) (5,1) (5,2) (5,3) (5,4) (5,5) (5,6) 

. . . . . . .
  — — — — —   
.|C 1 2 E|A|.
    — — —     
.|. 3 .|F|.|.
              
.|0 . . 4 .|.
  —   —       
.|D . E|S|B|.
  — — —   —   
. . . . . . .

Internal actions:
F L F F P F U F R F R F D B L B L B L F F G R F P F L L F F L F L F R F F L F D B L F F R P F R F L F L F D B R P F R F D B B B G F F L B B B B L F F R F F P F U F F F D B B B

Robot commands:
F L F F F R R F R F R F B L B L B L F F R F F L L F F L F L F R F F L F B L F F R F R F L F L F B R F R F B B B F F L B B B B L F F R F F F R R F F F B B B

Order summary:
bottle 0 to gate D, gate C, bottle 1 to gate E, bottle 3 to gate F, bottle 4 to gate B, gate A, bottle 2 to gate C

Bottle to gate mapping:
bottle 0 to gate D
bottle 1